# Introduction to AutoGen

**LangGraph** models workflows as **state graphs** with explicit nodes and edges (**02. LangGraph/**). **CrewAI** assigns **roles and tasks** in sequential or hierarchical crews. **AutoGen** takes a third path: **multi-agent conversation** — autonomous agents exchange messages until a task completes or a termination condition fires.

AutoGen excels when collaboration is naturally **dialogue-driven**: a writer and critic refining a draft, a planner delegating to a coder, or a human proxy approving tool use. Microsoft rewrote AutoGen in **0.4** as an **async, event-driven** stack with a layered **Core** + **AgentChat** API — this track uses **AgentChat 0.4+**, not legacy `pyautogen` 0.2.

This notebook defines AutoGen, contrasts it with LangGraph and CrewAI, previews the **AgentChat** mental model, maps the **03. AutoGen/** learning path, and runs a minimal async hello agent. Landscape context: **00. Frameworks/07. AutoGen**.

Recommended background: **01. Introduction to AI Orchestration Frameworks** and familiarity with async Python (`async def`, `await`).

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import asyncio
import importlib.metadata as im
import os

os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

for pkg in ("autogen-agentchat", "autogen-ext", "autogen-core"):
    try:
        print(f"{pkg}:", im.version(pkg))
    except im.PackageNotFoundError:
        print(f"{pkg}: not installed — pip install autogen-agentchat autogen-ext[openai]")

---

## 1. The Problem — When Conversation Beats a Graph

Fixed pipelines (LCEL, simple chains) run once top-to-bottom. **Agentic** work often needs **back-and-forth**:

```
user task ──► assistant drafts ──► critic feedback ──► assistant revises ──► user approves
                  ▲                              │
                  └──────────────────────────────┘
```

| Requirement | LangGraph | AutoGen AgentChat |
|-------------|-----------|-------------------|
| **Explicit control flow** | First-class (`StateGraph`) | Emergent from team + termination |
| **Multi-agent dialogue** | Model as nodes/subgraphs (**11**) | Native message passing |
| **Human approval** | `interrupt` + checkpoint (**09**) | `UserProxyAgent` + `input_func` (**04**) |
| **Code execution loops** | Tool nodes (**07**) | `CodeExecutorAgent` + teams (**07**) |
| **Rapid research prototyping** | More boilerplate | Fast two-agent / group chat setup |

AutoGen does not replace LangGraph — choose based on whether **conversation** or **declared graph** is the better mental model for your product.

---

## 2. What Is AutoGen?

**AutoGen** (Microsoft) is a framework for building **multi-agent LLM applications**. In **0.4+** it ships as multiple packages:

1. **`autogen-core`** — event-driven actor runtime (low-level).
2. **`autogen-agentchat`** — high-level **AgentChat** API: agents, teams, termination.
3. **`autogen-ext`** — model clients (OpenAI, Azure, etc.), code executors, extras.

AgentChat is the **successor** to the v0.2 `pyautogen` API. Legacy names like **ConversableAgent** and **GroupChat** map to new primitives — we cover mappings in **03** and **08**.

---

## 3. Multi-Agent Conversation Model

```
         ┌──────────────────────────────────────────┐
         │              Team (or single agent)         │
         │  RoundRobinGroupChat / SelectorGroupChat   │
         └──────────────────────────────────────────┘
              ▲                    ▲                    ▲
              │ messages           │                    │
    ┌─────────┴────────┐  ┌───────┴────────┐  ┌───────┴────────┐
    │ AssistantAgent   │  │ UserProxyAgent │  │ Specialist     │
    │ (LLM + tools)    │  │ (human input)  │  │ agents         │
    └─────────┬────────┘  └───────┬────────┘  └───────┬────────┘
              │                    │                    │
              └──────────► model_client ◄────────────────┘
                           (OpenAIChatCompletionClient)
```

| Concept | Meaning |
|---------|--------|
| **Agent** | Entity with a name, optional system message, tools |
| **Message** | `TextMessage`, tool calls, handoffs between agents |
| **Team** | Orchestrates which agent speaks next |
| **Termination** | Stops the run (`APPROVE`, max messages, external signal) |
| **`run` / `run_stream`** | Execute a task; returns `TaskResult` or async message stream |

---

## 4. AgentChat 0.4 Rewrite — What Changed

| v0.2 (`pyautogen`) | v0.4 AgentChat |
|--------------------|----------------|
| `llm_config={...}` | `model_client=OpenAIChatCompletionClient(...)` |
| `ConversableAgent` | `AssistantAgent`, `UserProxyAgent`, custom `BaseChatAgent` |
| `GroupChat` + `GroupChatManager` | `RoundRobinGroupChat`, `SelectorGroupChat`, etc. |
| Synchronous `initiate_chat` | `async` `await agent.run()` / `await team.run()` |
| `register_function` | Pass `tools=[...]` to `AssistantAgent` |
| Opaque chat history dict | Typed messages + `TaskResult` |

Install **Microsoft's** packages: `autogen-agentchat` and `autogen-ext` — not unrelated `pyautogen` releases after 0.2.34.

---

## 5. AutoGen vs LangGraph vs CrewAI

| | **AutoGen** | **LangGraph** | **CrewAI** |
|--|-------------|---------------|------------|
| **Core metaphor** | Agents in conversation | Nodes on a state graph | Crew roles + tasks |
| **Control** | Team manager + termination | Declared edges + routers | Process (seq / hierarchical) |
| **Async** | Native `async`/`await` | `ainvoke`, `astream` | Mixed; often sync wrappers |
| **HITL** | `UserProxyAgent` | `interrupt` + checkpointer | Human task inputs |
| **Production structure** | Add guardrails (**13**, **16**) | Strongest explicit audit trail | Demo-friendly crews |
| **Best fit** | Research, coding loops, dialogue | Regulated flows, checkpointing | Role-based task pipelines |

Many teams use **LangGraph for production routing** and **AutoGen for internal multi-agent experiments** — or embed AutoGen teams behind a FastAPI gateway.

---

## 6. Package Architecture

| Package | Role |
|---------|------|
| **`autogen-core`** | Cancellation, components, low-level messaging |
| **`autogen-agentchat`** | `AssistantAgent`, `UserProxyAgent`, teams, conditions |
| **`autogen-ext`** | `OpenAIChatCompletionClient`, code executors, MCP bridges |

Typical imports in this track:

```python
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
```

---

## 7. Mental Model — Task-Driven Runs

Unlike LangGraph's persistent **state object**, AgentChat is **task-driven**:

```
task string ──► agent.run(task) ──► [messages...] ──► TaskResult
                      │
                      └── run_stream ──► yield each message (Console UI)
```

Teams coordinate **multiple agents** on one task. Conversation history accumulates in the **team runtime** until termination. Stateful sessions: **12. Memory and Conversation State**.

---

## 8. Notes API Teaching Corpus

Examples across this track use a fictional **Notes API** (FastAPI, Alembic, JWT, pytest) so agents have realistic backend context — same corpus as **02. LangGraph/**:

In [ ]:
NOTES = {
    "n1": "The Notes API is built with FastAPI. Routes live under /notes.",
    "n2": "Run alembic upgrade head after pulling database migrations.",
    "n3": "JWT bearer tokens use Authorization: Bearer header.",
    "n4": "Pytest fixtures for DB setup belong in conftest.py.",
}

DOC_CHUNKS = [
    {"id": "c1", "text": NOTES["n1"]},
    {"id": "c2", "text": NOTES["n2"]},
    {"id": "c3", "text": NOTES["n3"]},
    {"id": "c4", "text": NOTES["n4"]},
    {"id": "c5", "text": "Use httpx.AsyncClient in FastAPI tests for async endpoints."},
]

print("Notes corpus:", list(NOTES.keys()), "| chunks:", [c["id"] for c in DOC_CHUNKS])

---

## 9. Your First Agent — Minimal Async Hello

A single `AssistantAgent` with one `run()` call. Requires a valid `OPENAI_API_KEY` (set in the setup cell):

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient


async def hello_autogen() -> None:
    model_client = OpenAIChatCompletionClient(
        model="gpt-4o-mini",
        api_key=OPENAI_API_KEY,
    )
    agent = AssistantAgent(
        name="notes_helper",
        model_client=model_client,
        system_message="You answer briefly about backend APIs.",
    )
    result = await agent.run(task="In one sentence, what is FastAPI?")
    print("stop_reason:", result.stop_reason)
    print("last message:", result.messages[-1].content)
    await model_client.close()


# In scripts: asyncio.run(hello_autogen())
await hello_autogen()

### 9.1 What Just Happened

1. **`OpenAIChatCompletionClient`** wraps the OpenAI chat API (replaces v0.2 `llm_config`).
2. **`AssistantAgent`** binds the client with a **system message**.
3. **`await agent.run(task=...)`** runs the full turn loop and returns **`TaskResult`**.
4. **`await model_client.close()`** releases HTTP connections — important in long-running apps.

Setup details and offline demos: **02. Setup and the AgentChat API**.

---

## 10. Offline Hello — No API Key Required

When you cannot call OpenAI, simulate the AgentChat message shape locally:

In [ ]:
from dataclasses import dataclass


@dataclass
class FakeTaskResult:
    stop_reason: str
    messages: list


async def offline_hello(task: str) -> FakeTaskResult:
    answer = (
        "FastAPI is a modern Python web framework for building APIs with "
        "type hints and automatic OpenAPI docs."
    )
    if "alembic" in task.lower():
        answer = NOTES["n2"]
    return FakeTaskResult(
        stop_reason="completed",
        messages=[{"role": "assistant", "content": answer}],
    )


offline = await offline_hello("What is Alembic used for?")
print(offline.messages[-1]["content"])

---

## 11. Teams Preview — Two Agents in Round Robin

Multi-agent collaboration uses **teams** (replaces v0.2 `GroupChat`). Full coverage in **05** and **08**:

In [ ]:
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat

# Sketch — uncomment when API key is set:
# async def preview_team() -> None:
#     client = OpenAIChatCompletionClient(model="gpt-4o-mini", api_key=OPENAI_API_KEY)
#     writer = AssistantAgent("writer", model_client=client, system_message="Draft concise answers.")
#     critic = AssistantAgent(
#         "critic",
#         model_client=client,
#         system_message="Reply APPROVE when the draft is good enough.",
#     )
#     team = RoundRobinGroupChat(
#         [writer, critic],
#         termination_condition=TextMentionTermination("APPROVE"),
#     )
#     result = await team.run(task="Explain JWT in one sentence.")
#     print(result.messages[-1].content)
#     await client.close()
# await preview_team()

print("Team pattern: RoundRobinGroupChat([writer, critic], termination_condition=...)")

---

## 12. Cross-Reference — LangGraph and LangChain

| AutoGen concept | LangGraph / LangChain analogue |
|-----------------|--------------------------------|
| `AssistantAgent` + tools | Tool-calling agent node (**02. LangGraph/06–07**) |
| `RoundRobinGroupChat` | Multi-agent supervisor (**02. LangGraph/11**) |
| `UserProxyAgent` | Human-in-the-loop interrupt (**02. LangGraph/09**) |
| `TextMentionTermination` | Conditional edge to END |
| `model_client` | `ChatOpenAI` (**01. LangChain/03**) |

Use **LangChain** for prompts and parsers; **LangGraph** when the graph is the product contract; **AutoGen** when agent dialogue is the primary abstraction.

---

## 13. When to Choose AutoGen

**Choose AutoGen** when:

- Tasks benefit from **multiple agents debating or refining** output
- You need **code execution loops** with a human proxy (**07**)
- **Conversation-driven** coordination fits better than drawing every edge
- You are prototyping in **AutoGen Studio** (**14**)

**Prefer LangGraph** when:

- Compliance requires a **visible state machine** and checkpoint resume
- Routing logic is **fixed and auditable**
- You already use **`create_agent`** and need custom nodes only

**Prefer CrewAI** when:

- Work decomposes into **role + task lists** with minimal custom code

---

## 14. Async-First Design

AgentChat 0.4 is **async-native**. In notebooks:

```python
async def main():
    result = await agent.run(task="...")

await main()  # top-level await in Jupyter
```

In FastAPI services, call the same coroutines from route handlers — do not block the event loop with sync `input()`. Patterns in **04** and **16**.

---

## 15. Installation

From the repo root with the project virtual environment activated:

```bash
pip install "autogen-agentchat" "autogen-ext[openai]"
```

Environment variable:

```bash
export OPENAI_API_KEY=sk-...
```

Or set `OPENAI_API_KEY` in the notebook setup cell. Deep setup walkthrough: **02. Setup and the AgentChat API**.

---

## 16. Design Principles to Carry Forward

1. **Agents have clear roles** — system messages are contracts, not decoration.
2. **Always set termination** — unbounded group chat is expensive.
3. **Close model clients** — treat them like DB connection pools.
4. **Prefer AgentChat 0.4 APIs** — do not copy v0.2 Stack Overflow snippets.
5. **Stream for UX** — `run_stream` + `Console` for interactive sessions (**02**).
6. **Layer guardrails in production** — timeouts, sandboxes, observability (**13–16**).

---

## 17. Pattern Map — Notebooks 02–16

| Pattern | Notebook |
|---------|----------|
| Setup + `model_client` | **02** |
| `AssistantAgent` + tools | **03**, **06** |
| Human approval | **04** |
| Writer + critic two-agent | **05** |
| `RoundRobinGroupChat` | **08**, **09** |
| Code execution | **07** |
| Termination guardrails | **13** |
| Production hardening | **16** |

Each builds on the async AgentChat API introduced here.

In [ ]:
import matplotlib.pyplot as plt

frameworks = ["CrewAI", "AutoGen", "LangGraph"]
dialogue_vs_structure = [7, 9, 4]  # illustrative: higher = more conversation-native

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.barh(frameworks, dialogue_vs_structure, color=["#55a868", "#4c72b0", "#c44e52"])
ax.set_xlabel("Conversation-native orchestration (illustrative 1–10)")
ax.set_title("AutoGen sits between role-based crews and explicit graphs")
ax.set_xlim(0, 12)
for bar, n in zip(bars, dialogue_vs_structure):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2, str(n), va="center")
plt.tight_layout()
plt.show()

---

## 18. Learning Path — This Series

```
01 Introduction ──► 02 Setup/AgentChat API ──► 03 AssistantAgent
     │
     ▼
04 UserProxy/HITL ──► 05 Two-agent patterns ──► 06 Tools
     │
     ▼
07 Code execution ──► 08 GroupChat teams ──► 09 Speaker selection
     │
     ▼
10 Custom roles ──► 11 Sequential/hierarchical ──► 12 Memory/state
     │
     ▼
13 Termination/guardrails ──► 14 AutoGen Studio ──► 15 Observability
     │
     ▼
16 Production patterns
```

Each notebook is **self-contained** but follows this sequence. Framework comparisons live in **`00. Frameworks/07. AutoGen`**. LangGraph checkpointing and graph interrupts are covered in **`02. LangGraph/09`**.

---

## 19. Common Beginner Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Using legacy `pyautogen` 0.2 imports | Wrong API, broken tutorials | Install `autogen-agentchat` 0.4+ |
| Forgetting `await` on `run()` | Coroutine never executes | Use `async def` + `await` or `asyncio.run()` |
| Sharing one `model_client` incorrectly | Leaked connections in prod | `await model_client.close()` when done |
| No termination condition on teams | Unbounded LLM spend | `TextMentionTermination` / `MaxMessageTermination` (**13**) |
| Blocking `input()` in async servers | Event loop stalls | Async `input_func` or queue-based HITL (**04**) |
| AutoGen for rigid state machines | Hard-to-audit flows | Use **LangGraph** when graph = contract |

---

## 20. Summary

**AutoGen 0.4 AgentChat** builds **multi-agent LLM applications** through **async message passing**: **agents** (especially `AssistantAgent` and `UserProxyAgent`) coordinated by **teams** with **termination conditions**. It complements **LangGraph** (explicit graphs) and **CrewAI** (role/task crews).

Key takeaways:

- v0.4 replaces `llm_config` with **`model_client`** and sync chats with **`await run()`**.
- **Teams** (`RoundRobinGroupChat`, etc.) replace v0.2 **GroupChat**.
- Choose AutoGen when **conversation** is the natural coordination model.
- Use the **Notes API** corpus for realistic backend agent demos.
- Patterns map to notebooks **02–16** in this track.

Demonstrations verified package imports, introduced the conversation model, ran a minimal hello agent (and offline variant), and previewed team composition.

Next: **02. Setup and the AgentChat API** — package layout, `OpenAIChatCompletionClient`, asyncio in notebooks, `Console` streaming, and offline demos.